# Evaluating Base, Prompted, and Fine-Tuned Phi-3 on Yoda Translation

This notebook evaluates three conditions on the same 20-example external test set:

1. **Base Phi-3** — the sentence is supplied without a style instruction.
2. **Base Phi-3 + system prompt** — the base model receives an explicit Yoda-style instruction.
3. **Fine-tuned Phi-3** — a previously trained LoRA adapter is loaded and evaluated without the system prompt.

The notebook uses deterministic generation, BERTScore, and chrF. Upload `yoda_external_test.csv` and your adapter folder to the Colab runtime before running the evaluation.

## Setup

Select a GPU runtime in Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!pip install -q transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 bitsandbytes==0.47.0 datasets==4.0.0 evaluate bert-score sacrebleu pandas

In [ ]:
import os
import torch
import evaluate
import numpy as np
import pandas as pd
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sacrebleu import corpus_chrf

torch.manual_seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
REPO_ID = "microsoft/Phi-3-mini-4k-instruct"
ADAPTER_PATH = '/content/drive/MyDrive/344/code/Yoda/local-phi3-mini-yoda-adapter'
TEST_CSV_PATH = '/content/drive/MyDrive/344/code/Yoda/yoda_external_test.csv'
#this file doesn't exist the first time you run it
#when you do this eval for GenZ change the results file name so it doesn't orverride
RESULTS_PATH = "/content/drive/MyDrive/344/code/Yoda/yoda_evaluation_results_demo.csv"

## Load the quantized base model

The configuration follows the original fine-tuning notebook and loads Phi-3 in 4-bit NF4 format.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    device_map="auto",
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

print(f"Model memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model memory footprint: 2206.3 MB


## Load and verify the external test set

These examples were created outside the published 720-row training dataset, so they can be used to evaluate an adapter that was trained on all published rows.

In [ ]:
test_dataset = load_dataset(
    "csv",
    data_files=TEST_CSV_PATH,
    split="train",
)

required_columns = {"sentence", "translation_extra"}
assert required_columns.issubset(test_dataset.column_names)
assert len(test_dataset) == 20, f"Expected 20 test examples, found {len(test_dataset)}"
assert len(set(test_dataset["sentence"])) == 20, "Test sentences must be unique."

test_sentences = test_dataset["sentence"]
references = test_dataset["translation_extra"]

display(test_dataset.to_pandas())

,id,sentence,translation_extra,category
0,1,The professor uploaded the lecture slides befo...,"Before lunch, the lecture slides the professor...",declarative
1,2,Our robotics team finished the prototype last ...,"Last night, the prototype our robotics team fi...",declarative
2,3,The campus bus arrived just before the rain be...,"Just before the rain began, the campus bus arr...",declarative
3,4,My roommate left the apartment key on the kitc...,"On the kitchen counter, the apartment key my r...",declarative
4,5,The astronomy club will watch the meteor showe...,"Tonight, the meteor shower the astronomy club ...",future
5,6,The new software detected three errors in my p...,"Three errors in my program, the new software d...",declarative
6,7,The study group did not finish the final chapter.,"Finish the final chapter, the study group did ...",negative
7,8,I cannot remember the password for the laborat...,Remember the password for the laboratory compu...,negative
8,9,You should charge your laptop before the exami...,"Before the examination, charge your laptop, yo...",advice
9,10,We must reserve a room for tomorrow's meeting.,"A room for tomorrow's meeting, reserve we must.",obligation


## Prompting and generation helpers

In [ ]:
YODA_SYSTEM_PROMPT = """
You rewrite ordinary English sentences in Yoda's distinctive speaking style.

Rules:
- Preserve the original meaning.
- Use Yoda-like inverted sentence structure.
- Keep the response concise.
- You may occasionally use expressions such as "Hmm" or "Hrrmm."
- Return only the rewritten sentence.
- Do not explain your answer.
""".strip()


def build_messages(sentence, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": sentence})
    return messages


def generate_response(model, tokenizer, sentence, system_prompt=None, max_new_tokens=64):
    messages = build_messages(sentence, system_prompt)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    prompt_length = inputs["input_ids"].shape[1]
    model.eval()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = output[0, prompt_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## Condition 1: base Phi-3 without a style prompt

In [ ]:
base_predictions = [
    generate_response(base_model, tokenizer, sentence)
    for sentence in test_sentences
]



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
base_predictions

['The sentence provided is already in the past tense, indicating that the action of uploading the lecture slides was completed before lunch. No change is needed to reflect the past tense.',
 "Congratulations on completing the prototype for your robotics team! This is a significant milestone in the development process. The completion of a prototype is a critical step as it allows you to test the design, functionality, and performance of your robot in a controlled environment. It'ieves you with valuable insights that",
 'The campus bus arrived just before the rain began.',
 "I understand that you're concerned about the security of your apartment. It'ieves to ensure that your roommate is aware of the importance of keeping the key secure. You might want to have a conversation with them about the potential risks of leaving keys unattended and discuss a plan for key management that",
 'The instruction is a simple directive indicating that the members of the astronomy club are planning to obs

## Condition 2: base Phi-3 with the Yoda system prompt

In [ ]:
prompted_predictions = [
    generate_response(
        base_model,
        tokenizer,
        sentence,
        system_prompt=YODA_SYSTEM_PROMPT,
    )
    for sentence in test_sentences
]

prompted_predictions[:3]

['"Before lunch, the lecture slides were uploaded by the professor, yes?"',
 'Last night, the prototype was finished by our robotics team, Hrrmm.',
 'Just before the rain began, the campus bus arrived, it did.']

## Condition 3: load and evaluate the fine-tuned LoRA adapter

The adapter is loaded onto the same quantized base model. The test input contains only the sentence, matching the one-turn format used by the original Yoda fine-tuning notebook.

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

fine_tuned_predictions = [
    generate_response(fine_tuned_model, tokenizer, sentence)
    for sentence in test_sentences
]

fine_tuned_predictions[:3]

['Uploaded before lunch, the lecture slides the professor did. Yes, hrrrm.',
 'Finished the prototype last night, our robotics team did. Yes, hrrrm.',
 'Just before the rain began, the campus bus arrived. Hmm.']

## Automatic evaluation

BERTScore estimates semantic similarity to the reference. chrF measures character n-gram overlap and is sensitive to wording and word forms. Because many valid Yoda translations are possible, neither metric should be interpreted as a perfect measure of style.

In [ ]:
bertscore = evaluate.load("bertscore")


def evaluate_predictions(name, predictions, references):
    scores = bertscore.compute(
        predictions=predictions,
        references=references,
        lang="en",
        model_type="distilbert-base-uncased",
    )

    per_example_f1 = np.asarray(scores["f1"])
    chrf = corpus_chrf(predictions, [references]).score

    return {
        "model": name,
        "BERTScore F1": float(per_example_f1.mean()),
        "chrF": float(chrf),
        "per_example_f1": per_example_f1,
    }


evaluation_results = [
    evaluate_predictions("Base Phi-3", base_predictions, references),
    evaluate_predictions("Base Phi-3 + system prompt", prompted_predictions, references),
    evaluate_predictions("Fine-tuned Phi-3", fine_tuned_predictions, references),
]

summary = pd.DataFrame([
    {k: v for k, v in result.items() if k != "per_example_f1"}
    for result in evaluation_results
]).sort_values("BERTScore F1", ascending=False)

display(summary.style.format({"BERTScore F1": "{:.4f}", "chrF": "{:.2f}"}))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

,model,BERTScore F1,chrF
2,Fine-tuned Phi-3,0.9035,78.58
1,Base Phi-3 + system prompt,0.9034,75.05
0,Base Phi-3,0.7832,36.63


## Per-example comparison

In [ ]:
score_lookup = {result["model"]: result["per_example_f1"] for result in evaluation_results}

comparison = pd.DataFrame({
    "sentence": test_sentences,
    "reference": references,
    "base_prediction": base_predictions,
    "prompted_prediction": prompted_predictions,
    "fine_tuned_prediction": fine_tuned_predictions,
    "base_bertscore": score_lookup["Base Phi-3"],
    "prompted_bertscore": score_lookup["Base Phi-3 + system prompt"],
    "fine_tuned_bertscore": score_lookup["Fine-tuned Phi-3"],
})

score_columns = ["base_bertscore", "prompted_bertscore", "fine_tuned_bertscore"]
winner_names = {
    "base_bertscore": "Base",
    "prompted_bertscore": "System prompt",
    "fine_tuned_bertscore": "Fine-tuned",
}
comparison["winner"] = comparison[score_columns].idxmax(axis=1).map(winner_names)
comparison["fine_tuning_vs_base"] = comparison["fine_tuned_bertscore"] - comparison["base_bertscore"]
comparison["prompting_vs_base"] = comparison["prompted_bertscore"] - comparison["base_bertscore"]

display(comparison)
print("Per-example winners:")
display(comparison["winner"].value_counts().rename_axis("condition").to_frame("wins"))

,sentence,reference,base_prediction,prompted_prediction,fine_tuned_prediction,base_bertscore,prompted_bertscore,fine_tuned_bertscore,winner,fine_tuning_vs_base,prompting_vs_base
0,The professor uploaded the lecture slides befo...,"Before lunch, the lecture slides the professor...",The sentence provided is already in the past t...,"""Before lunch, the lecture slides were uploade...","Uploaded before lunch, the lecture slides the ...",0.809835,0.846673,0.875100,Fine-tuned,0.065265,0.036838
1,Our robotics team finished the prototype last ...,"Last night, the prototype our robotics team fi...",Congratulations on completing the prototype fo...,"Last night, the prototype was finished by our ...","Finished the prototype last night, our robotic...",0.757001,0.893212,0.885618,System prompt,0.128618,0.136211
2,The campus bus arrived just before the rain be...,"Just before the rain began, the campus bus arr...",The campus bus arrived just before the rain be...,"Just before the rain began, the campus bus arr...","Just before the rain began, the campus bus arr...",0.971253,0.951941,0.945994,Base,-0.025260,-0.019312
3,My roommate left the apartment key on the kitc...,"On the kitchen counter, the apartment key my r...",I understand that you're concerned about the s...,"Left the apartment key on the kitchen counter,...","The apartment key on the kitchen counter, my r...",0.715903,0.909783,0.893494,System prompt,0.177591,0.193880
4,The astronomy club will watch the meteor showe...,"Tonight, the meteor shower the astronomy club ...",The instruction is a simple directive indicati...,"""Watch the meteor shower the astronomy club wi...","Watch the meteor shower tonight, the astronomy...",0.783329,0.913260,0.927061,Fine-tuned,0.143732,0.129932
5,The new software detected three errors in my p...,"Three errors in my program, the new software d...",The sentence provided is a simple statement th...,"Detected three errors in your program, the new...","Detected three errors in my program, the new s...",0.729738,0.934100,0.914622,System prompt,0.184885,0.204363
6,The study group did not finish the final chapter.,"Finish the final chapter, the study group did ...","The sentence ""The study group did not finish t...","Did not finish the final chapter, the study gr...","Did not finish the final chapter, the study gr...",0.798762,0.926728,0.891082,System prompt,0.092320,0.127966
7,I cannot remember the password for the laborat...,Remember the password for the laboratory compu...,If you've forgotten the password for the labor...,"Cannot remember the password, I cannot.","The password for the laboratory computer, I ca...",0.784985,0.904974,0.901531,System prompt,0.116546,0.119989
8,You should charge your laptop before the exami...,"Before the examination, charge your laptop, yo...",To ensure that your laptop is fully charged an...,"Charge your laptop, you should.","Charge your laptop, you should, before the exa...",0.762470,0.912374,0.943961,Fine-tuned,0.181491,0.149904
9,We must reserve a room for tomorrow's meeting.,"A room for tomorrow's meeting, reserve we must.","To reserve a room for tomorrow's meeting, you ...","Reserve a room, we must for tomorrow's meeting.","Reserve a room for tomorrow's meeting, we must.",0.783487,0.942061,0.952894,Fine-tuned,0.169408,0.158574


Per-example winners:


,wins
condition,
Fine-tuned,10
System prompt,9
Base,1


## Automated analysis summary

The following cell reports the best aggregate condition, score changes relative to the unprompted base model, and per-example wins. Use this evidence when writing the final interpretation.

In [ ]:
summary_by_name = summary.set_index("model")
best_bert = summary.iloc[0]["model"]
best_chrf = summary.sort_values("chrF", ascending=False).iloc[0]["model"]
base_bert = summary_by_name.loc["Base Phi-3", "BERTScore F1"]
prompted_bert = summary_by_name.loc["Base Phi-3 + system prompt", "BERTScore F1"]
tuned_bert = summary_by_name.loc["Fine-tuned Phi-3", "BERTScore F1"]
wins = comparison["winner"].value_counts()

print("AUTOMATED ANALYSIS")
print("-" * 60)
print(f"Highest mean BERTScore: {best_bert}")
print(f"Highest corpus chrF: {best_chrf}")
print(f"Prompting change vs. base BERTScore: {prompted_bert - base_bert:+.4f}")
print(f"Fine-tuning change vs. base BERTScore: {tuned_bert - base_bert:+.4f}")
print(f"Fine-tuning change vs. prompted BERTScore: {tuned_bert - prompted_bert:+.4f}")
print("Per-example wins:", wins.to_dict())
print()
print("Interpretation caution: a single reference cannot represent every valid")
print("Yoda-style rewrite. Inspect disagreements between BERTScore and chrF,")
print("and do not treat either metric as a complete measure of stylistic quality.")

AUTOMATED ANALYSIS
------------------------------------------------------------
Highest mean BERTScore: Fine-tuned Phi-3
Highest corpus chrF: Fine-tuned Phi-3
Prompting change vs. base BERTScore: +0.1202
Fine-tuning change vs. base BERTScore: +0.1203
Fine-tuning change vs. prompted BERTScore: +0.0001
Per-example wins: {'Fine-tuned': 10, 'System prompt': 9, 'Base': 1}

Interpretation caution: a single reference cannot represent every valid
Yoda-style rewrite. Inspect disagreements between BERTScore and chrF,
and do not treat either metric as a complete measure of stylistic quality.


## Save the results

In [ ]:
comparison.to_csv(RESULTS_PATH, index=False)
print(f"Saved detailed results to {RESULTS_PATH}")

Saved detailed results to /content/drive/MyDrive/344/code/Yoda/yoda_evaluation_results_demo.csv


## Final analysis

Using the generated tables, write 200–300 words addressing:

1. Which condition achieved the highest BERTScore and chrF?
2. Did prompting or fine-tuning produce the larger improvement over the base model?
3. How often did each condition win on individual examples?
4. Why might BERTScore and chrF disagree?
5. Why can a valid prediction receive a weak score against a single reference?
6. Does the evidence justify fine-tuning for this style-transfer task?